# Reference chemical lists

Builds the combined compound table used throughout the study: known mammary
carcinogens (**MC**) and suspected mammary carcinogens (**SMC**, drawn from
endocrine-disruptor databases). Compounds are keyed on **InChIKey**, so that
the same identifier joins the chemical lists, the LINCS metadata and the
results tables.

**Produces** `../../data/chemicals/Chemicals_concat.csv`, the input to the
LINCS extraction (`analyses/02`).

In [1]:
import pandas as pd

CHEMICALS = "../../data/chemicals"

In [2]:
# One row per compound: duplicate InChIKeys come from synonym and source
# duplication in the upstream databases, not from distinct structures.
smc = pd.read_csv(f"{CHEMICALS}/mat_bdd_query_ED_all_chem_30-10-2023.csv")
smc = smc.drop_duplicates(subset=["INCHIKEY"])
smc["Chem_type"] = "SMC"

mc = pd.read_csv(f"{CHEMICALS}/Merged_MC_chemlist.csv")
mc = mc.drop_duplicates(subset=["INCHIKEY"])
mc["Chem_type"] = "MC"

print(f"MC: {mc.shape[0]} compounds | SMC: {smc.shape[0]} compounds")

MC: 268 compounds | SMC: 1626 compounds


In [3]:
# MC first, so that a compound present in both lists keeps the MC label: a known
# carcinogen is not re-labelled as merely suspected.
chemicals = pd.concat([mc, smc], ignore_index=True)
chemicals = chemicals.drop(
    columns=["idchemicals", "synonyms", "PUBCHEM_CID", "CAS"], errors="ignore"
)
chemicals = chemicals.drop_duplicates(subset="INCHIKEY", keep="first").reset_index(
    drop=True
)

print(chemicals["Chem_type"].value_counts().to_string())
print(f"total: {chemicals.shape[0]} distinct compounds")
chemicals.head()

Chem_type
SMC    1551
MC      268
total: 1819 distinct compounds


,SMILES,CHEMICAL_NAME,SOURCE_TOX,IUPAC_NAME,INCHIKEY,Chem_type,ED_ACTIVE
0,C=CC1=CC=CC=C1,Styrene,Borrel,styrene,PPBRXRYQALVLMV-UHFFFAOYSA-N,MC,NaN
1,NC1=C(Cl)C=C(CC2=CC(Cl)=C(N)C=C2)C=C1,"4,4'-Methylenebis(2-chloroaniline)",Borrel,4-[(4-amino-3-chlorophenyl)methyl]-2-chloroani...,IBOFVQJTBBUKMU-UHFFFAOYSA-N,MC,NaN
2,C1=CC=C2C(=C1)/C(=N/O)/C=CN2O.Cl,4-Hydroxyaminoquinoline 1-oxide hydrochloride,Borrel,(NE)-N-(1-hydroxyquinolin-4-ylidene)hydroxylam...,QSXDAZCGWHRQME-VRTOBVRTSA-N,MC,NaN
3,CN1C(N)=NC2=C1C=C(C=N2)C1=CC=CC=C1,"2-Amino-1-methyl-6-phenylimidazo [4,5-b]pyridine",Borrel,"1-methyl-6-phenylimidazo[4,5-b]pyridin-2-amine",UQVKZNNCIHJZLS-UHFFFAOYSA-N,MC,NaN
4,CCCCCN(N=O)C(N)=O,1-Amyl-1-nitrosourea,Borrel,1-nitroso-1-pentylurea,YYTNAQDGJQPZFU-UHFFFAOYSA-N,MC,NaN


In [4]:
chemicals.to_csv(f"{CHEMICALS}/Chemicals_concat.csv", index=False)